# 1. Imports and configuration
Set up the environment for EDA.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path().resolve().parent
RAW_DATA_PATH = BASE_DIR / "raw" / "dataset.csv"


# 2. Load dataset
Load data/raw/dataset.csv and display a small sample.

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
df.head()


# 3. Dataset dimensions
Check number of rows and columns.

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


# 4. Column inspection
Check exact column names and data types.

In [ ]:
df.dtypes


# 5. Missing-value analysis
Count missing values and calculate percentages.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})


# 6. Duplicate analysis
Check for exact duplicates and duplicate product names.

In [ ]:
exact_dupes = df.duplicated().sum()
dup_names = df.duplicated(subset=['Product Name']).sum()
print(f"Exact duplicate rows: {exact_dupes}")
print(f"Duplicate product names: {dup_names}")


# 7. Product-name analysis
Analyze unique names, lengths, and most frequent items.

In [ ]:
unique_names = df['Product Name'].nunique()
print(f"Unique product names: {unique_names}")
print("\nMost frequent names:")
print(df['Product Name'].value_counts().head(5))

name_lens = df['Product Name'].str.len()
print(f"\nName length - Min: {name_lens.min()}, Max: {name_lens.max()}, Mean: {name_lens.mean():.2f}")


# 8. Category analysis
Distribution of products across categories.

In [ ]:
print(f"Unique categories: {df['Category'].nunique()}")
print("\nTop Categories (Count):")
print(df['Category'].value_counts().head(10))

print("\nTop Categories (%):")
print(df['Category'].value_counts(normalize=True).head(10) * 100)


# 9. Quantity analysis
Look at the unique quantity values and representations.

In [ ]:
print(f"Unique quantity representations: {df['Quantity'].nunique()}")
print("\nTop Quantities:")
print(df['Quantity'].value_counts().head(10))


# 10. Original-price analysis
Look at stats for 'Original Price (Rs.)'.

In [ ]:
df['Original Price (Rs.)'].describe(percentiles=[.25, .5, .75, .95, .99])


# 11. Discount analysis
Analyze the 'Discount' string column (e.g., '10% OFF').

In [ ]:
discount_numeric = df['Discount'].astype(str).str.replace('% OFF', '', regex=False).str.replace('% off', '', regex=False).str.strip()
discount_numeric = pd.to_numeric(discount_numeric, errors='coerce')
discount_numeric.describe(percentiles=[.25, .5, .75, .95, .99])


# 12. Discounted-price analysis
Stats for 'Discounted Price (Rs.)'.

In [ ]:
df['Discounted Price (Rs.)'].describe(percentiles=[.25, .5, .75, .95, .99])


# 13. Price consistency checks
Verify discounted price <= original price.

In [ ]:
dp_less = (df['Discounted Price (Rs.)'] < df['Original Price (Rs.)']).sum()
dp_eq = (df['Discounted Price (Rs.)'] == df['Original Price (Rs.)']).sum()
dp_gt = (df['Discounted Price (Rs.)'] > df['Original Price (Rs.)']).sum()
print(f"Discounted < Original: {dp_less}")
print(f"Discounted == Original: {dp_eq}")
print(f"Discounted > Original: {dp_gt}")


# 14. Product examples
Display 20 representative rows.

In [ ]:
df.sample(20, random_state=42)


# 15. Data-quality findings

### Strengths
- **High Coverage**: Product names, categories, and quantities have 100% non-null coverage.
- **Price Consistency**: Out of ~25k rows, there are exactly 0 cases where the discounted price is mathematically greater than the original price. This means the pricing logic in the raw data is remarkably consistent.
- **Rich Variety**: 21,822 unique products spanning 39 categories provide a robust catalog.

### Missing Fields
- `Original Price (Rs.)` and `Discounted Price (Rs.)` each have 2,564 missing values (~10%).
- `Discount` has 38 missing values (0.15%).

### Inconsistent Fields
- **Categories**: Several categories are promotional rather than descriptive product types (e.g., `Minimum 30% Off`, `bigbasket`, `Rs. 100 to 199`). These must be normalized or re-categorized.
- **Quantities**: Stored as raw text strings (e.g., `80 pcs`, `180 ml - Tetra Pack`). These require regex parsing to split into `quantity_value` and `quantity_unit`.

### Suspicious Values
- **Zero Prices**: The minimum Original Price and Discounted Price are both `0.0`, which likely denotes an error or a free item anomaly that should not be a primary retail product.
- **Outlier Prices**: The maximum Original Price is `59,750.52`, which may be valid for some categories but warrants scrutiny.

### Likely Cleaning Requirements
- Discard or flag the 63 exact duplicate rows.
- Drop rows with missing prices (or impute if appropriate).
- Parse `Quantity` into numeric values and units.
- Clean `Discount` by stripping the '% OFF' strings and converting to numeric floats.
- Remap promotional categories to an `Other` or `Promotional` category.

### Direct Mapping
- `Product Name` -> `name`
- `Original Price (Rs.)` -> `price`
- `Discounted Price (Rs.)` -> `sale_price`

### Required Transformations
- `Category` -> Requires mapping to standard schema values.
- `Quantity` -> Must be split into `quantity_value` (numeric) and `quantity_unit` (text).
- `Discount` -> If kept, must be transformed to a float; `is_on_sale` must be derived from `sale_price < price`.
- Missing fields in schema (e.g., `brand`, `is_organic`) will need to be derived or left null initially.